# Cost plot

OpenAI list $/1M tokens vs a placeholder self-host price (`SELF_HOSTED_PRICE_PER_MILLION_TOKENS` in code).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# USD per 1M tokens (input+output averaged for simplicity — adjust for your mix)
PRICE_MINI_IN = 0.15
PRICE_MINI_OUT = 0.60
PRICE_4O_IN = 2.50
PRICE_4O_OUT = 10.0
SELF_HOSTED_PER_MILLION = float(os.environ.get("SELF_HOSTED_PRICE_PER_MILLION_TOKENS", "1.2"))

_p = Path.cwd().resolve()
ROOT = next((a for a in [_p, *_p.parents] if (a / "pyproject.toml").is_file()), _p)
summary_path = ROOT / "notebooks" / "finetune_classifier" / "eval_summary.json"
avg_tokens_per_request = 1200
if summary_path.exists():
    data = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Loaded eval_summary:", data)

daily_requests = np.array([1_000, 10_000, 100_000])
tokens_per_day = daily_requests * avg_tokens_per_request
millions_per_month = tokens_per_day * 30 / 1e6

def monthly_cost_mini(millions: np.ndarray) -> np.ndarray:
    # assume 70% input / 30% output tokens
    return millions * (0.7 * PRICE_MINI_IN + 0.3 * PRICE_MINI_OUT)

def monthly_cost_4o(millions: np.ndarray) -> np.ndarray:
    return millions * (0.7 * PRICE_4O_IN + 0.3 * PRICE_4O_OUT)

def monthly_cost_self(millions: np.ndarray) -> np.ndarray:
    return millions * SELF_HOSTED_PER_MILLION

c_mini = monthly_cost_mini(millions_per_month)
c_4o = monthly_cost_4o(millions_per_month)
c_self = monthly_cost_self(millions_per_month)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(daily_requests, c_mini, label="gpt-4o-mini (API)", marker="o")
ax.plot(daily_requests, c_4o, label="gpt-4o (API)", marker="s")
ax.plot(daily_requests, c_self, label=f"Self-hosted (~${SELF_HOSTED_PER_MILLION}/1M tok)", marker="^")
ax.set_xlabel("Daily classification requests")
ax.set_ylabel("Monthly cost (USD)")
ax.set_xscale("log")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

cross = np.where(c_self < c_mini)[0]
if len(cross):
    print("Self-hosted cheaper than gpt-4o-mini at daily volume >=", daily_requests[cross[0]])
else:
    print("At modeled token prices, API mini remains cheaper than self-hosted in this band; raise volume or lower GPU $/token.")